# Compare our hourly $ET_0$ with `AgERA5` $ET_0$

## First download `AgERA5` data using the CDS API

In [ ]:
import s3fs
import os
import cdsapi
import pandas as pd
from tqdm.notebook import tqdm

# Whether to re-download AgERA5 data
redownload = False

s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'

out_dir = f'{s3_base_path}/input-data/raw-data/agera5'
start_date = '2001-01-01'
end_date = '2020-12-31'

# Load in csv of site locations/names
site_df = pd.read_csv(f'{s3_base_path}/sites/site-locations.csv')

# If re-downloading, remove old files first
if redownload:
    for site in site_df['site_id']:
        out_path = os.path.join(out_dir, f"{site}.csv")
        s3.rm(out_path)

def download_agera5_et(site_id, lat, lon, start_date, end_date, out_dir):
    """Download AgERA5 reference evapotranspiration for one location.

    Notes
    -----
    Requires a configured CDS API key in ~/.cdsapirc.
    AgERA5 reference ET is daily and in mm/day.

    If the CDS form uses slightly different field names for point extraction,
    open the dataset page, select the same options manually, and copy the
    generated API request into this function.
    """

    csv_name = f"{site_id}.csv"
    out_path = os.path.join(out_dir, csv_name)

    dataset = "sis-agrometeorological-indicators-timeseries"
    request = {
        "variable": ["reference_evapotranspiration"],
        "date": [f"{start_date}/{end_date}"],
        "data_format": "csv",
        "location": {"longitude": lon, "latitude": lat}
    }

    client = cdsapi.Client(quiet=True)
    client.retrieve(dataset, request).download(target=csv_name)

    # Move local file to S3
    s3.put(csv_name, out_path)
    os.remove(csv_name)

    return

# Download for all sites
for i in tqdm(site_df.index):
    site = site_df.loc[i, 'site_id']
    lat = site_df.loc[i, 'lat']
    lon = site_df.loc[i, 'lon']
    if not redownload:
        continue

    download_agera5_et(site_id=site, lat=lat, lon=lon, out_dir=out_dir,
                       start_date=start_date, end_date=end_date)

In [ ]:
# Combine each individual csv into one dataframe
agera5_et = pd.DataFrame(index=pd.date_range(start_date, end_date),
                        columns=site_df['site_id'])
for i in site_df.index:
    site = site_df.loc[i, 'site_id']
    lat = site_df.loc[i, 'lat']
    lon = site_df.loc[i, 'lon']
    tmp_df = pd.read_csv(f'{s3_base_path}/input-data/raw-data/agera5/{site}.csv', index_col=0,
                         parse_dates=True)
    # Print comparison of target lat/lon
    print(f"{site}: lat={tmp_df.loc['2001-01-01', 'latitude']} vs {lat},")
    print(f"{site}: lon={tmp_df.loc['2001-01-01', 'longitude']} vs {lon}")
    print()
    agera5_et[site] = tmp_df['ReferenceET_PenmanMonteith_FAO56']

agera5_et.to_csv(f'{s3_base_path}/input-data/processed-data/climate/agera5_ref_et.csv')

## Plot scatter of AgERA5 vs our aggregated hourly ET for each site

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load in hourly reference ET
ref_et = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/ref_et.csv',
                     index_col=0, parse_dates=True)
# Adjust time from UTC to central time (UTC-6)
ref_et = ref_et.shift(-6, freq='h')

# Resample to daily
ref_et = ref_et.resample('D').sum()

# Drop first/last incomplete records
ref_et.drop(['2000-12-31', '2020-12-31'], inplace=True)

In [ ]:
# Create scatter plot for each site
for site in ref_et.columns:

    x = agera5_et.loc[ref_et.index, site].values
    y = ref_et[site].values

    rmse = np.sqrt(np.mean((y - x) ** 2))
    bias = np.mean(y - x)
    r = np.corrcoef(x, y)[0, 1]
    # Calculate slope of best fit line
    slope = np.polyfit(x, y, 1)[0]

    max_val = np.nanmax([x.max(), y.max()])
    lim = (0, max_val * 1.05)

    fig, ax = plt.subplots(figsize=(5, 5), tight_layout=True)

    ax.scatter(x, y, s=8, alpha=0.25, color='k')
    ax.plot(lim, lim, linestyle="--", linewidth=1)

    ax.set(xlim=lim, ylim=lim, xlabel='AgERA5 (mm/d)', ylabel='Our ET (mm/d)',
           title=site)

    text = (
        f"RMSE = {rmse:.2f} mm/day\n"
        f"Bias = {bias:.2f} mm/day\n"
        f"r = {r:.2f}\n"
        f"Slope = {slope:.2f}\n"
    )

    ax.text(0.05, 0.95, text, transform=ax.transAxes, va="top", ha="left")

    fig.savefig(f'plots/ET_comparison_{site}.png', bbox_inches='tight', dpi=300)

    if site != 'Flanagan':
        plt.close(fig)

## Compare climate metrics across sites

In [ ]:
# Load in data
ref_et = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/ref_et.csv', index_col=0, parse_dates=True)
ppt_df = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/precip.csv', index_col=0, parse_dates=True)

# Calculate daily, monthly and annual metrics
ppt_daily = ppt_df.resample("D").sum()
et_daily = ref_et.resample("D").sum()

ppt_monthly = ppt_daily.resample("ME").sum()
et_monthly = et_daily.resample("ME").sum()

ppt_annual = ppt_daily.resample("YE").sum()
et_annual = et_daily.resample("YE").sum()

In [ ]:
# Calculate site-level metrics
metrics = pd.DataFrame(index=ppt_df.columns)

# Mean annual precipitation and reference ET
metrics["MAP_mm_yr"] = ppt_annual.mean()
metrics["ET0_mm_yr"] = et_annual.mean()

# Aridity index: lower = drier, higher = wetter
metrics["aridity_index"] = metrics["MAP_mm_yr"] / metrics["ET0_mm_yr"]

# Precipitation seasonality index (Walsh and Lawler style)
monthly_clim = ppt_monthly.groupby(ppt_monthly.index.month).mean()
metrics["precip_seasonality"] = (
    monthly_clim.sub(metrics["MAP_mm_yr"] / 12, axis=1).abs().sum()
    / metrics["MAP_mm_yr"]
)

# Growing season metrics: April–September
growing_months = [4, 5, 6, 7, 8, 9]

ppt_gs = ppt_daily[ppt_daily.index.month.isin(growing_months)].resample("YE").sum()
et_gs = et_daily[et_daily.index.month.isin(growing_months)].resample("YE").sum()

metrics["growing_season_P_mm"] = ppt_gs.mean()
metrics["growing_season_ET0_mm"] = et_gs.mean()
metrics["growing_season_deficit_mm"] = metrics["growing_season_ET0_mm"] - metrics["growing_season_P_mm"]

In [ ]:
# Plot annual averages
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(metrics['MAP_mm_yr'], metrics['ET0_mm_yr'])

# Add point labels
for site, row in metrics.iterrows():
    ax.annotate(site, (row['MAP_mm_yr'], row['ET0_mm_yr']), xytext=(4, 4),
        textcoords="offset points", fontsize=8)

ax.set(xlabel='Mean annual precip (mm/yr)', ylabel='Annual reference ET (mm/yr)')

In [ ]:
# Plot aridity vs precipitation seasonality
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(metrics['aridity_index'], metrics['precip_seasonality'])

# Add point labels
for site, row in metrics.iterrows():
    ax.annotate(site, (row['aridity_index'], row['precip_seasonality']), xytext=(4, 4),
        textcoords="offset points", fontsize=8)

ax.set(xlabel='Pseudo aridity index', ylabel='Precipitation seasonality index')

In [ ]:
# Plot growing-season P vs growing-season ET0
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(metrics['growing_season_P_mm'], metrics['growing_season_ET0_mm'])

# Add point labels
for site, row in metrics.iterrows():
    ax.annotate(site, (row['growing_season_P_mm'], row['growing_season_ET0_mm']), xytext=(4, 4),
        textcoords="offset points", fontsize=8)

ax.set(xlabel='Growing season precip (mm)', ylabel='Growing season ET (mm)')

# Add 1:1 line
lim_max = max(
    metrics["growing_season_P_mm"].max(),
    metrics["growing_season_ET0_mm"].max(),
) * 1.05

ax.plot([0, lim_max], [0, lim_max], color='k', ls='--')
ax.set_xlim(0, lim_max)
ax.set_ylim(0, lim_max)

In [ ]:
# Monthly climatology
monthly_clim = ppt_monthly.groupby(ppt_monthly.index.month).mean()

winter_months = [12, 1, 2]
summer_months = [6, 7, 8]

winter_p = monthly_clim.loc[winter_months].sum()
summer_p = monthly_clim.loc[summer_months].sum()
annual_p = monthly_clim.sum()

metrics["winter_p_frac"] = winter_p / annual_p
metrics["summer_p_frac"] = summer_p / annual_p

# Directional seasonality index:
# positive = summer-wet, negative = winter-wet
metrics["summer_minus_winter_frac"] = (
    metrics["summer_p_frac"] - metrics["winter_p_frac"]
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(metrics["summer_minus_winter_frac"], metrics["precip_seasonality"])

# Add point labels
for site, row in metrics.iterrows():
    ax.annotate(site, (row['summer_minus_winter_frac'], row['precip_seasonality']), xytext=(4, 4),
        textcoords="offset points", fontsize=8)
ax.text(0.02, 0.02, 'Winter-wet', ha='left', va='bottom', transform=ax.transAxes, fontstyle='italic')
ax.text(0.98, 0.02, 'Summer-wet', ha='right', va='bottom', transform=ax.transAxes, fontstyle='italic')

ax.axvline(0, linestyle="--")

ax.set(xlabel="Summer minus winter precipitation fraction", ylabel="Precipitation seasonality index",
       xlim=(-0.7, 0.7))